# SLEAP Pipeline — Local
**Moita Lab · Champalimaud Foundation**
Rodrigo Garrido

---
## AVISO CRÍTICO — GPU NVIDIA

Este pipeline corre redes neurais profundas. O desempenho depende fortemente do hardware:

| Situação | Tempo estimado por vídeo |
|---|---|
| **GPU NVIDIA** (ex: RTX 2060 ou superior) | **2–10 minutos** |
| **CPU apenas** (sem GPU) | **1–4 horas** |

> **Sem GPU, o pipeline funciona — mas é muito lento.**  
> Para verificar se o teu PC tem GPU compatível, corre a célula de validação (Passo 2) antes de iniciar.

**Requisitos mínimos com GPU:**
- GPU NVIDIA com pelo menos 4 GB VRAM
- Drivers NVIDIA atualizados (versão ≥ 450)
- CUDA Toolkit 11.3 (instalado automaticamente pelo `setup_env.bat`)

---

## Descrição

Pipeline local para **pose estimation de moscas** (*Drosophila*) com SLEAP.ai.

Combina dois sistemas de tracking:
- **Bonsai** — centroide da mosca (coordenadas normalizadas 0–1)
- **SLEAP** — 8 keypoints relativos ao centroide (Head, Thorax, Abdomen, Left, Right, LeftWing, RightWing, Top)

Output: um CSV por mosca com todas as posições normalizadas (0–1) relativas à arena.

**Referências:**  
SLEAP.ai — https://doi.org/10.1038/s41592-022-01426-1  
Moita Lab — https://moitalab.org/

## Setup (apenas uma vez por PC)

Antes de usar este notebook pela primeira vez, executa `setup_env.bat`  
(está na mesma pasta deste ficheiro).

O script faz automaticamente:
1. Instala o ambiente conda `sleap_env` com SLEAP, TensorFlow e CUDA
2. Regista o kernel `Python (sleap_env)` no Jupyter
3. Descarrega o modelo treinado do GitHub (Moita Lab)

> **Certifica-te que tens este notebook aberto com o kernel `Python (sleap_env)`**  
> (canto superior direito no JupyterLab / VS Code)

## Ordem de execução

1. **Passo 1** — Preenche os caminhos na célula de configuração
2. **Passo 2** — Corre a validação (verifica pastas e GPU)
3. **Passo 3** — Corre o pipeline (inferência SLEAP + post-processing)

---
## Passo 1 — Configuração

Edita os caminhos abaixo antes de correr qualquer célula.

In [ ]:
import os

# ================================================================
# CONFIGURAÇÃO — edita estes caminhos antes de correr o pipeline
# ================================================================

# >>> DIRETÓRIO A DEFINIR AQUI: pasta raiz do teu experimento <<<
EXPERIMENT_ROOT = r"E:\Champalimaud_Project_MoitaLab\Colab"

# Pasta do modelo SLEAP (criada pelo setup_env.bat na mesma pasta deste notebook)
MODEL_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "Sleap_Model")

# ----------------------------------------------------------------
# Estrutura de subpastas (altera apenas se a tua estrutura for diferente)
# ----------------------------------------------------------------
VIDEO_FOLDER   = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "CropRaw")
ARENAS_FOLDER  = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "Arenas")
TRACKED_FOLDER = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "Tracked")
OUTPUT_FOLDER  = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "Pose")
TEMP_FOLDER    = os.path.join(EXPERIMENT_ROOT, "PostProcessing", "Temp_SLP")

print("Configuração carregada.")
print(f"  Experimento : {EXPERIMENT_ROOT}")
print(f"  Modelo      : {MODEL_PATH}")

---
## Passo 2 — Validação

Verifica que todas as pastas existem e deteta a GPU disponível.

In [ ]:
import tensorflow as tf

print("=" * 55)
print(" VALIDAÇÃO DE CAMINHOS")
print("=" * 55)

errors = []
checks = [
    ("Vídeos (CropRaw)",   VIDEO_FOLDER),
    ("Arenas",             ARENAS_FOLDER),
    ("Tracked (Bonsai)",   TRACKED_FOLDER),
    ("Modelo SLEAP",       MODEL_PATH),
]

for label, path in checks:
    if os.path.exists(path):
        n = len(os.listdir(path))
        print(f"  OK  {label}: {path} ({n} items)")
    else:
        errors.append(f"  ERRO  {label}: {path}")
        print(f"  ERRO  {label}: {path} -- NAO ENCONTRADO")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(TEMP_FOLDER,   exist_ok=True)
print(f"  OK  Output (Pose)   : {OUTPUT_FOLDER}")
print(f"  OK  Temp (.slp)     : {TEMP_FOLDER}")

print()
print("=" * 55)
print(" VERIFICAÇÃO DE GPU")
print("=" * 55)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"  GPU detectada: {len(gpus)} dispositivo(s)")
    for g in gpus:
        print(f"    - {g.name}")
    print("  Pipeline correra em modo GPU (rapido).")
else:
    print("  AVISO: Nenhuma GPU NVIDIA detectada.")
    print("  O pipeline correra em CPU -- pode demorar horas por video.")

print()
if errors:
    print("EXISTEM ERROS NOS CAMINHOS. Corrige antes de continuar.")
    raise RuntimeError("Caminhos invalidos — ver mensagens acima.")
else:
    print("Todas as verificacoes passaram. Podes correr o Passo 3.")

---
## Passo 3 — Pipeline

Corre esta célula para processar todos os vídeos na pasta `CropRaw`.  
Vídeos já processados (`.slp` existente) são automaticamente saltados.

In [ ]:
import os, sys, glob, subprocess
import sleap
import pandas as pd
import numpy as np
from PIL import Image

node_map = {
    'L': 'Left', 'R': 'Right', 'H': 'Head', 'Trx': 'Thorax',
    'Abd': 'Abdomen', 'Lw': 'LeftWing', 'Rw': 'RightWing', 'T': 'Top'
}

# ------------------------------------------------------------------
# 1. SLEAP Inference
# ------------------------------------------------------------------
print("=" * 55)
print(" SLEAP INFERENCE")
print("=" * 55)

video_files = sorted(glob.glob(os.path.join(VIDEO_FOLDER, "*.avi")) +
                     glob.glob(os.path.join(VIDEO_FOLDER, "*.mp4")))
if not video_files:
    raise RuntimeError(f"Nenhum vídeo (.avi/.mp4) encontrado em: {VIDEO_FOLDER}")

print(f"  {len(video_files)} video(s) encontrado(s).\n")

for video in video_files:
    c_id    = os.path.splitext(os.path.basename(video))[0]
    slp_out = os.path.join(TEMP_FOLDER, f"{c_id}.predictions.slp")

    if os.path.exists(slp_out):
        print(f"  [skip]  {c_id}  (ja processado)")
        continue

    print(f"  [track] {c_id} ...")
    result = subprocess.run(
        ["sleap-track", video, "--model", MODEL_PATH,
         "-o", slp_out, "--no-empty-frames"],
        capture_output=False
    )
    if result.returncode != 0:
        print(f"  [ERRO]  sleap-track falhou para {c_id}")

# ------------------------------------------------------------------
# 2. Post-processing: SLEAP + Bonsai
# ------------------------------------------------------------------
print()
print("=" * 55)
print(" POST-PROCESSING: SLEAP + BONSAI")
print("=" * 55)

slp_files = sorted(glob.glob(os.path.join(TEMP_FOLDER, "*.predictions.slp")))
if not slp_files:
    raise RuntimeError(f"Nenhum .slp encontrado em: {TEMP_FOLDER}")

for slp_path in slp_files:
    c_id = os.path.basename(slp_path).replace(".predictions.slp", "")

    # Nome do CSV Bonsai: substituir _crop por _tracked
    tracked_id = c_id.replace("_crop", "_tracked")
    csv_path = os.path.join(TRACKED_FOLDER, f"{tracked_id}.csv")
    if not os.path.exists(csv_path):
        print(f"  [skip] Sem CSV Bonsai para {c_id}  (esperado: {tracked_id}.csv)")
        continue

    print(f"  Processando: {c_id}")

    session_prefix = c_id.split('-fly')[0].split('_fly')[0]
    arena_candidates = [
        os.path.join(ARENAS_FOLDER, f"{c_id.replace('_crop', '')}.png"),
        os.path.join(ARENAS_FOLDER, f"{session_prefix}.png"),
    ]
    arena_img = next((p for p in arena_candidates if os.path.exists(p)), None)

    if arena_img:
        with Image.open(arena_img) as img:
            w_arena, h_arena = float(img.size[0]), float(img.size[1])
    else:
        tried = [os.path.basename(p) for p in arena_candidates]
        print(f"    Arena nao encontrada (tentei: {tried}) -- usando 1280x1024.")
        w_arena, h_arena = 1280.0, 1024.0

    df_t   = pd.read_csv(csv_path)
    labels = sleap.load_file(slp_path)

    col_x = [c for c in df_t.columns if 'X' in c.upper() and ('CENTROID' in c.upper() or len(c) == 1)][0]
    col_y = [c for c in df_t.columns if 'Y' in c.upper() and ('CENTROID' in c.upper() or len(c) == 1)][0]

    data = []
    for frame in labels:
        f_idx = frame.frame_idx
        c_row = df_t[df_t['FrameIndex'] == f_idx]
        if c_row.empty:
            continue

        cx_px = c_row[col_x].values[0] * w_arena
        cy_px = c_row[col_y].values[0] * h_arena
        row   = {'FrameIndex': f_idx}

        if len(frame.instances) == 0:
            for name in node_map.values():
                row[f'{name}.Position.X'] = np.nan
                row[f'{name}.Position.Y'] = np.nan
                row[f'{name}.Confidence'] = 0.0
        else:
            for inst in frame.instances:
                for node in labels.skeleton.nodes:
                    pt   = inst[node.name]
                    name = node_map.get(node.name, node.name)
                    if pt is not None:
                        norm_x = (cx_px + (pt.x - 64.0)) / w_arena
                        norm_y = (cy_px + (pt.y - 64.0)) / h_arena
                        row[f'{name}.Position.X'] = max(0.0, min(1.0, norm_x))
                        row[f'{name}.Position.Y'] = max(0.0, min(1.0, norm_y))
                        row[f'{name}.Confidence'] = pt.score
                    else:
                        row[f'{name}.Position.X'] = np.nan
                        row[f'{name}.Position.Y'] = np.nan
                        row[f'{name}.Confidence'] = 0.0
        data.append(row)

    if data:
        out_csv = os.path.join(OUTPUT_FOLDER, f"{c_id}_pose.csv")
        pd.DataFrame(data).sort_values('FrameIndex').to_csv(out_csv, index=False, na_rep='NaN')
        print(f"    Guardado: {os.path.basename(out_csv)}")
    else:
        print(f"    Sem dados para {c_id}.")

print()
print("Pipeline concluido!")